In [ ]:
import pandas as pd
from preprocessing.geneva_stroke_unit_preprocessing.utils import create_registry_case_identification_column
from tqdm import tqdm

In [ ]:
perfusion_parameters_path = '/Users/jk1/temp/opsum_end/imaging_extraction/perfusion_parameters_20260120.csv'
perfusion_images_meta_data_path = '/Users/jk1/temp/opsum_end/imaging_extraction/dicom_metadata_extraction_20260121_185934.csv'
manual_extraction_path = '/Users/jk1/stroke_datasets/random_subset_for_imaging_extraction.xlsx'
stroke_registry_data_path = '/Users/jk1/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
perfusion_parameters_df = pd.read_csv(perfusion_parameters_path)
perfusion_images_meta_data_df = pd.read_csv(perfusion_images_meta_data_path, dtype={'acquisition_time': str})
manually_extracted_perfusion_df = pd.read_excel(manual_extraction_path)


In [ ]:
stroke_registry_df = pd.read_excel(stroke_registry_data_path)
stroke_registry_df['patient_id'] = stroke_registry_df['Case ID'].apply(lambda x: x[8:-4])
stroke_registry_df['EDS_last_4_digits'] = stroke_registry_df['Case ID'].apply(lambda x: x[-4:])
stroke_registry_df['case_admission_id'] = create_registry_case_identification_column(stroke_registry_df)

In [ ]:
# merge on 'patient_id' and 'file_path'
merged_df = pd.merge(perfusion_parameters_df, perfusion_images_meta_data_df, on=['patient_id', 'file_path', 'acquisition_date'], how='left')
perfusion_parameters_df = merged_df

In [ ]:
# create a new paramenter_name column by combining parameter_type and threshold columns
perfusion_parameters_df['parameter_name'] = perfusion_parameters_df.apply(
    lambda row: f"{row['parameter_type']}_{row['threshold']}", axis=1
)
# remove spaces from parameter_name column
perfusion_parameters_df['parameter_name'] = perfusion_parameters_df['parameter_name'].str.replace(' ', '_')
# remove trailing ".0" from parameter_name column with regex
perfusion_parameters_df['parameter_name'] = perfusion_parameters_df['parameter_name'].str.replace(r'\.0$', '', regex=True)
# replace lower than symbol with 'lt' and greater than symbol with 'gt' in parameter_name column
perfusion_parameters_df['parameter_name'] = perfusion_parameters_df['parameter_name'].str.replace('<', 'lt_').str.replace('>', 'gt_')
# convert all to lowercase
perfusion_parameters_df['parameter_name'] = perfusion_parameters_df['parameter_name'].str.lower()

In [ ]:
# explicit loop without helper for clearer debugging; also duplicate rows when multiple admissions share the same closest date difference
expanded_rows = []
date_format = "%Y%m%d"

for _, row in tqdm(perfusion_parameters_df.iterrows(), total=len(perfusion_parameters_df)):
    patient_id = row['patient_id']
    acquisition_date = row['acquisition_date']
    patient_registry_df = stroke_registry_df[stroke_registry_df['patient_id'].astype(str) == str(patient_id)].copy()
    if patient_registry_df.empty:
        row_copy = row.copy()
        row_copy['case_admission_id'] = None
        row_copy['date_diff'] = None
        row_copy['arrival_at_hospital'] = None
        expanded_rows.append(row_copy)
        continue
    patient_registry_df['date_diff'] = (pd.to_datetime(patient_registry_df['Arrival at hospital'], format=date_format) - pd.to_datetime(acquisition_date, format=date_format)).dt.total_seconds() / (24 * 3600)
    patient_registry_df['abs_date_diff'] = patient_registry_df['date_diff'].abs()
    min_abs_diff = patient_registry_df['abs_date_diff'].min()
    closest_rows = patient_registry_df[patient_registry_df['abs_date_diff'] == min_abs_diff]
    for _, match in closest_rows.iterrows():
        row_copy = row.copy()
        row_copy['case_admission_id'] = match['case_admission_id']
        row_copy['date_diff'] = match['date_diff']
        row_copy['arrival_at_hospital'] = match['Arrival at hospital']
        expanded_rows.append(row_copy)

perfusion_parameters_df = pd.DataFrame(expanded_rows)

In [ ]:
# join manually extracted data
# rename columns 'T10', 'T8', 'T6', 'T4', 'CBF' to match parameter_name format
rename_dict = {
    'T10': 'tmax_gt_10',
    'T8': 'tmax_gt_8',
    'T6': 'tmax_gt_6',
    'T4': 'tmax_gt_4',
    'CBF': 'cbf_lt_30',
    '1st brain imaging date': 'acquisition_date',
    '1st brain imaging time': 'acquisition_time'
}
manually_extracted_perfusion_df = manually_extracted_perfusion_df.rename(columns=rename_dict)

In [ ]:
value_vars = ['tmax_gt_10', 'tmax_gt_8', 'tmax_gt_6', 'tmax_gt_4', 'cbf_lt_30']
manually_extracted_perfusion_df = manually_extracted_perfusion_df.melt(id_vars=['case_admission_id', 'patient_id', 'acquisition_date', 'acquisition_time'],
                                         value_vars=value_vars, var_name='parameter_name',
                                         value_name='value')

# drop rows where value is nan
manually_extracted_perfusion_df = manually_extracted_perfusion_df.dropna(subset=['value'])

In [ ]:
# rename columns in perfusion_parameters_df to match manually_extracted_perfusion_df
perfusion_parameters_df = perfusion_parameters_df.rename(columns={'volume': 'value'})

# drop unnecessary columns
perfusion_parameters_df = perfusion_parameters_df[['case_admission_id', 'patient_id', 'acquisition_date', 'acquisition_time', 'parameter_name', 'value']]

In [ ]:
perfusion_parameters_df.shape, manually_extracted_perfusion_df.shape

In [ ]:
# create oveall combined dataframe true concat
combined_perfusion_df = pd.concat([perfusion_parameters_df, manually_extracted_perfusion_df], ignore_index=True)

In [ ]:
# save to csv with timestamp
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
# output_path = f'/Users/jk1/temp/opsum_end/imaging_extraction/combined_perfusion_parameters_{timestamp}.csv'
# combined_perfusion_df.to_csv(output_path, index=False)